# Bias-correction mode sanity check (hot/calm/sunny scenario)

This notebook reproduces an ad-hoc manual test run during the `climate-tab` /
`feat/climate-tool` merge (see PR #87) to sanity-check the four
`bias_correction` modes newly wired up in `scripts/pool_physics.py`:

- `none` — original bulk-transfer evaporation (baseline, always-on default)
- `evap_multiplier` — scales the bulk-transfer coefficient by `CE_MULTIPLIER`
- `wind_floor` — imposes a minimum effective wind speed (`EVAP_WIND_FLOOR_MS`)
  for evaporation, modeling natural convection on calm days
- `penman` — Penman (1948) combination equation, radiation + wind driven

**Motivation**: the original `worktree-bias-correction` branch found that the
baseline (`none`) bulk-transfer evaporation term under-predicts cooling on
hot, calm, sunny days (the exact conditions where evaporative cooling should
matter most). `evap_multiplier` and `wind_floor` are simple empirical fixes
for that. `penman` is a from-first-principles alternative that should also
increase cooling on this kind of day (per its own docstring: "sunny calm days
produce MORE evaporation than the linear-wind bulk formula would predict").

**Finding**: `evap_multiplier` and `wind_floor` both behave as expected
(more evaporative cooling than baseline). `penman` currently does the
**opposite** — it predicts *less* cooling than baseline on this scenario,
which contradicts its own design intent. This notebook isolates that result
so it can be investigated/re-tuned later. Nothing in production currently
calls any mode other than `none` (see `ACTIVE_BIAS_CORRECTION = 'none'` in
`pool_physics.py`), so this is a pre-existing tuning question, not a live bug.


## Setup

Import the physics module directly from `scripts/`.

In [1]:
import sys
from pathlib import Path

# Notebook lives in experiments/, physics module lives in scripts/
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

from pool_physics import (
    pool_thermal_balance_step,
    CE_MULTIPLIER,
    EVAP_WIND_FLOOR_MS,
)

print(f"CE_MULTIPLIER      = {CE_MULTIPLIER}")
print(f"EVAP_WIND_FLOOR_MS = {EVAP_WIND_FLOOR_MS} m/s")


CE_MULTIPLIER      = 1.25
EVAP_WIND_FLOOR_MS = 1.5 m/s


## Scenario: hot, calm, sunny day

A Waco-in-August type day — warm pool, hot air, near-still wind, full sun.
This is exactly the "baseline under-cools" scenario the bias-correction
work was meant to address.

In [2]:
scenario = dict(
    T_pool_C       = 30.0,   # warm pool (86 F)
    T_air_C        = 36.0,   # hot air   (97 F)
    RH_pct         = 35.0,   # dry
    wind_speed_ms  = 0.5,    # nearly calm
    solar_MJm2_day = 26.0,   # strong summer solar load
    depth_m        = 2.0,
    ground_temp_C  = 22.0,
    bottom_u_Wm2K  = 1.0,
    dt_days        = 1.0,
)

print("Scenario (hot / calm / sunny):")
for k, v in scenario.items():
    print(f"  {k:16s} = {v}")


Scenario (hot / calm / sunny):
  T_pool_C         = 30.0
  T_air_C          = 36.0
  RH_pct           = 35.0
  wind_speed_ms    = 0.5
  solar_MJm2_day   = 26.0
  depth_m          = 2.0
  ground_temp_C    = 22.0
  bottom_u_Wm2K    = 1.0
  dt_days          = 1.0


## Run all four bias-correction modes

In [3]:
modes = ['none', 'evap_multiplier', 'wind_floor', 'penman']
results = {}

for mode in modes:
    T_new, fluxes = pool_thermal_balance_step(
        **scenario,
        bias_correction=mode,
    )
    results[mode] = (T_new, fluxes)
    print(f"--- bias_correction={mode!r} ---")
    print(f"  Q_evap_Wm2 = {fluxes['Q_evap_Wm2']:8.2f}   "
          f"Q_total_Wm2 = {fluxes['Q_total_Wm2']:8.2f}   "
          f"dT_C = {fluxes['dT_C']:+.4f}   "
          f"T_pool_new_C = {T_new:.4f}")


--- bias_correction='none' ---
  Q_evap_Wm2 =   -25.38   Q_total_Wm2 =   267.32   dT_C = +2.7588   T_pool_new_C = 32.7588
--- bias_correction='evap_multiplier' ---
  Q_evap_Wm2 =   -31.73   Q_total_Wm2 =   260.98   dT_C = +2.6933   T_pool_new_C = 32.6933
--- bias_correction='wind_floor' ---
  Q_evap_Wm2 =   -76.15   Q_total_Wm2 =   216.56   dT_C = +2.2349   T_pool_new_C = 32.2349
--- bias_correction='penman' ---
  Q_evap_Wm2 =    -0.33   Q_total_Wm2 =   292.37   dT_C = +3.0173   T_pool_new_C = 33.0173


## Compare against baseline (`none`)

In [4]:
baseline_evap = results['none'][1]['Q_evap_Wm2']
baseline_dT   = results['none'][1]['dT_C']

print(f"{'mode':16s} {'Q_evap_Wm2':>12s} {'dT_C':>10s} {'delta_vs_none':>16s} {'more_cooling?':>14s}")
for mode in modes:
    fluxes = results[mode][1]
    delta = fluxes['Q_evap_Wm2'] - baseline_evap
    more_cooling = 'YES' if fluxes['Q_evap_Wm2'] < baseline_evap else ('baseline' if mode == 'none' else 'NO <-- unexpected')
    print(f"{mode:16s} {fluxes['Q_evap_Wm2']:12.2f} {fluxes['dT_C']:10.4f} {delta:16.2f} {more_cooling:>14s}")


mode               Q_evap_Wm2       dT_C    delta_vs_none  more_cooling?
none                   -25.38     2.7588             0.00       baseline
evap_multiplier        -31.73     2.6933            -6.35            YES
wind_floor             -76.15     2.2349           -50.77            YES
penman                  -0.33     3.0173            25.05 NO <-- unexpected


## Result

- `evap_multiplier` and `wind_floor` both produce a **more negative**
  `Q_evap_Wm2` (more evaporative cooling) than `none`, as intended.
- `penman` produces a **less negative** (or even positive-shifted) `Q_evap_Wm2`
  than `none` on this hot/calm/sunny scenario — i.e. it predicts *less*
  cooling, the opposite of its stated design goal.

## Suspected cause (not yet fixed)

Looking at `penman_evap_Wm2()` in `pool_physics.py`:

1. It uses `saturation_vapor_pressure_kPa(T_air_C)` for the vapor-pressure
   slope `Δ` (correct, Δ is normally evaluated at air/mean temp), but it
   estimates net radiation `Rn` from *incoming solar only*, ignoring net
   longwave loss and (more importantly) the pool's own reflection/absorption
   already accounted for elsewhere in `pool_thermal_balance_step`. On a hot
   day the pool is close to (or above) air temperature, so the real `Rn`
   available for evaporation is smaller than raw incoming solar suggests —
   this could be under-driving the radiation term.
2. The aerodynamic term `Ea_mm_day` uses the *same* `(e_s_pool - e_a)` vapor
   deficit as the baseline model, scaled by a wind function `f_u` that has a
   **large constant offset** (`6.43 * (1 + 0.536*u)` ⇒ even at `u=0`,
   `f_u ≈ 6.43`, vs. the baseline's linear `EVAP_BASE * wind_speed_ms` which
   goes to **zero** as wind → 0). At `wind_speed_ms = 0.5` this actually
   pushes the aerodynamic term *up*, but the `Δ/(Δ+γ)` weighting
   (~0.78 radiation / 0.22 wind at 36 °C per the docstring) means the
   radiation term dominates — so an under-estimated `Rn` most likely
   explains the net under-cooling.
3. Unit/scale check worth doing separately: confirm `E_mm_day → W m⁻²`
   conversion and the `Rn` proxy against a reference Penman implementation
   (e.g. FAO-56) with the exact same inputs, to rule out a straightforward
   sign or unit-scaling bug rather than a physical modeling gap.

This is flagged for follow-up tuning — not resolved here. Nothing in
production currently uses `penman` (`ACTIVE_BIAS_CORRECTION = 'none'`).